# Phase 5 Cross-Seed Cue-Regime Sweep — K=1, β=10 (Colab, read-only over snapshots)

**Active phase:** 5
**Purpose:** Cue-regime sweep at the confirmed-best operating point (β=10, K=1, γ=0.5 per [report 057](https://github.com/Dypatterson/Neuro-AI/blob/main/reports/057_phase5_cross_seed_beta_sweep_K1.md)). The grid is `binding_noise_std × content_distortion`; for each cell, the aggregator records paired ΔE plus basin-membership and ordering counts.

**Per-cell metrics** (per GPT's recommendation to aggregate more than ΔE):
- `mean ΔE_raw` + 95% CI + seeds positive
- `ΔE / 5.5e-3 floor` ratio
- Ordering counts: `role < content < random`, `role < content`, `random_lowest` (the pathology metric — report 057 found random produces lowest energy at 9/10 seeds with K=1, β=10)
- Per-condition basin hit rate (argmax similarity == role_target_idx?)
- Per-condition mean role-target rank (1-indexed in similarity ordering)
- Per-condition entropy / max_w / mean energy

**Grid** (24 cells per seed):
- `binding_noise_std ∈ {0.01, 0.05, 0.10, 0.20}` (4)
- `content_distortion ∈ {0.0, 0.2, 0.4, 0.6, 0.8, 1.0}` (6)

**Decision rule** (informal, will surface in the aggregator's markdown):
- If any cell has `ΔE/floor ≥ 1.0` AND `basin hit rate ≥ 0.3` → cue operating point was the bottleneck; rerun full headline at that cell.
- If `ΔE` stays sub-floor but basin hit rate improves notably anywhere → option 3 (reformulate to basin-membership) becomes leading path.
- If basin hit rate is ~0 across all cells → option 1 (lower-D redesign) is the cleanest path.

**Wall-time estimate:** ~15-20 min total (10 seeds launched in parallel via subprocess.Popen, mirroring scripts/colab_phase5_headline_n10.ipynb). Outputs per-seed JSONs + a cross-seed aggregate JSON + markdown.

This is a **drill-down**, not a graduation experiment. No retuning of β/K/γ/ε/τ/formulation.


In [ ]:
# 1. Clone repo at the patched commit (must include --cue-regime-sweep mode).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -6

import subprocess, sys
markers = [
    ('--cue-regime-sweep',          'scripts/phase5_frozen_snapshot_audit.py',  'harness cue-regime mode'),
    ('_run_headline_cue_regime_sweep', 'scripts/phase5_frozen_snapshot_audit.py', 'cue-regime sweep core'),
    ('_basin_diagnostics',          'scripts/phase5_frozen_snapshot_audit.py',  'basin diagnostics helper'),
    ('frac_role_lt_content_lt_random', 'scripts/aggregate_cue_sweep.py',        'cross-seed aggregator'),
]
for marker, fpath, label in markers:
    r = subprocess.run(['grep', '-n', '-e', marker, fpath], capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'MISSING'
    print(f'  [{status}] {label}: {marker} in {fpath}')
    if r.returncode != 0:
        sys.exit(1)


In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'
print('drive results root:', DRIVE_RESULTS)


In [ ]:
# 3. Locate the 10 A1' snapshots.
import os
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'

snapshot_paths = {}
missing = []
for s in SEEDS:
    p = f'{DRIVE_RESULTS}/phase5_headline_substrate_seed{s}/snapshots/phase3_phase4_w4_step1800.pt'
    if os.path.exists(p):
        snapshot_paths[s] = p
        print(f'  seed {s:>3}: ok')
    else:
        missing.append(s)
        print(f'  seed {s:>3}: MISSING at {p}')

if missing:
    print(f'\n!!! missing: {missing}')


In [ ]:
# 4. Parent-CPU sanity. Do NOT init CUDA in parent.
!nvidia-smi --query-gpu=name,memory.total --format=csv | head -3


In [ ]:
# 5. PARALLEL cross-seed cue-regime sweep — launch 10 workers at once.
# Mirrors the pattern from colab_phase5_headline_n10.ipynb cells 7/10:
# subprocess.Popen per seed, poll loop every 30s. Parent must NOT init
# CUDA (we never call torch in this cell — each worker initializes its
# own CUDA context per CLAUDE.md memory colab_workflow.md).
#
# Each worker runs the harness's 24-cell sweep in-process. With 10
# workers contending for one GPU, expected wall-time ~15-20 min on T4
# or A100 (vs ~120 min sequential). GPU memory: ~300MB per worker for
# 1064 atoms at D=4096, so 10 workers ~3GB — well under T4's 16GB.
import subprocess, time, os, json
from pathlib import Path

RUN_TAG = 'phase5_cross_seed_cue_sweep'
N_CUES = 200
K_MAIN = 1
GAMMA = 0.5
BETA = 10.0
BNS_GRID = '0.01,0.05,0.10,0.20'
CD_GRID  = '0.0,0.2,0.4,0.6,0.8,1.0'

out_root = Path(f'reports/{RUN_TAG}')
out_root.mkdir(parents=True, exist_ok=True)
log_root = Path(f'reports/{RUN_TAG}/logs')
log_root.mkdir(parents=True, exist_ok=True)

# Parent env, copied to each worker.
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'

def launch(seed, snapshot):
    out_path = out_root / f'seed{seed}.json'
    log_path = log_root / f'seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [
        'python', 'scripts/phase5_frozen_snapshot_audit.py',
        '--snapshot', snapshot,
        '--output', str(out_path),
        '--cue-regime-sweep',
        '--cue-regime-binding-noise', BNS_GRID,
        '--cue-regime-content-distortion', CD_GRID,
        '--cue-regime-beta', str(BETA),
        '--cue-regime-n-cues', str(N_CUES),
        '--headline-k-main', str(K_MAIN),
        '--headline-gamma', str(GAMMA),
        '--device', 'cuda',
    ]
    print(f'  launching seed {seed}')
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    return proc, logf, out_path

print(f'launching {len(snapshot_paths)} parallel workers...')
procs = [(seed, *launch(seed, snap)) for seed, snap in snapshot_paths.items()]
t0 = time.time()
done_seeds = {}
remaining = list(range(len(procs)))
while remaining:
    still = []
    for i in remaining:
        seed, proc, logf, out_path = procs[i]
        rc = proc.poll()
        if rc is None:
            still.append(i)
        else:
            logf.close()
            mins = (time.time() - t0) / 60.0
            if rc == 0 and out_path.exists():
                done_seeds[seed] = out_path
                # Quick preview from this worker's JSON.
                with open(out_path) as f:
                    d = json.load(f)
                cells_list = d['cue_regime_sweep']['cells']
                best_dE = max(cells_list, key=lambda c: c['mean_delta_e_raw'])
                best_hit = max(cells_list, key=lambda c: c['per_condition_basin_hit_rate']['role'])
                print(f'  seed {seed:>3} done at {mins:.1f} min — '
                      f'best ΔE bns={best_dE["binding_noise_std"]}, cd={best_dE["content_distortion"]} '
                      f'(ΔE={best_dE["mean_delta_e_raw"]:+.5f}); '
                      f'best hit_role={best_hit["per_condition_basin_hit_rate"]["role"]:.2f} '
                      f'at bns={best_hit["binding_noise_std"]}, cd={best_hit["content_distortion"]}')
            else:
                print(f'  seed {seed:>3} FAILED at {mins:.1f} min (rc={rc}); see {log_root}/seed{seed}.log')
    remaining = still
    if remaining:
        time.sleep(30)

print(f'\n[total] {(time.time() - t0) / 60:.1f} min for {len(done_seeds)} successful seeds')
per_seed_jsons = done_seeds


In [ ]:
# 6. Run cross-seed aggregator.
import subprocess, os
env = {**os.environ, 'PYTHONPATH': '/content/Neuro-AI/src'}
out_root = 'reports/phase5_cross_seed_cue_sweep'
agg_path = f'{out_root}/cross_seed_aggregate.json'
r = subprocess.run([
    'python', 'scripts/aggregate_cue_sweep.py',
    '--sweep-root', out_root,
    '--output', agg_path,
], env=env, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)


In [ ]:
# 7. Print the aggregator's markdown.
with open('reports/phase5_cross_seed_cue_sweep/cross_seed_aggregate.md') as f:
    print(f.read())


In [ ]:
# 8. Copy results to Drive.
import shutil, os
DRIVE_RESULTS = '/content/drive/MyDrive/neuro-ai/results'
dst = f'{DRIVE_RESULTS}/phase5_cross_seed_cue_sweep'
os.makedirs(dst, exist_ok=True)
shutil.copytree('reports/phase5_cross_seed_cue_sweep', dst, dirs_exist_ok=True)
print(f'copied to {dst}')
!ls -la {dst} | head -25
